# RPD01 - SO-101 Teleoperation  (SO-101 - LeRobot - ROCm)

### Lab Description

The foundation of imitation learning is **teleoperation**: a *leader* arm you move by hand drives a *follower* arm in real time. In this lab you connect both SO-101 arms, work out which USB serial port maps to which arm, verify that their calibration files are loaded into the motors, and run `lerobot-teleoperate` so the follower mirrors the leader. Everything runs inside a ROCm Docker container on an **AMD-only** machine (no NVIDIA/CUDA). This teleoperation setup is exactly what RPD02 uses to record demonstrations.

## Lab Overview

| Step | Topic | Key Concepts |
|------|-------|-------------|
| 1 | Verify the Environment | PyTorch, ROCm, GPU availability |
| 2 | Identify USB Ports | `/dev/ttyACM*`, leader vs follower mapping |
| 3 | Verify Calibration Files | Calibration JSON, homing offsets, joint ranges |
| 4 | Run Teleoperation | `lerobot-teleoperate`, calibration path override |
| 5 | Verify Success | Real-time leader-follower mirroring |

#### Recommended Hardware

- Two SO-101 arms (one leader, one follower) with Feetech STS3215 servos
- USB hub connecting both arms to the host machine

#### Software Environment

Docker container built from `Dockerfile` with [LeRobot](https://github.com/huggingface/lerobot) v0.5.0+ and ROCm PyTorch.\
Launch with `run.sh` (GPU + USB devices mounted).

## Goals

1. **Understand Teleoperation**: Grasp how a leader arm controls a follower arm in real time.
2. **Identify USB Ports**: Determine which serial device maps to which arm.
3. **Verify Calibration**: Confirm calibration files are loaded correctly.
4. **Run Teleoperation**: Execute `lerobot-teleoperate` and handle the calibration prompt.
5. **Validate the Setup**: Confirm the follower mirrors the leader's movements.


## 1. Verify the Environment

Confirm that PyTorch and GPU (ROCm) are available inside the container.

In [4]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.9.1+rocm7.11.0
CUDA available: True


## 2. Identify USB Ports

Each arm connects via a USB-to-serial adapter. We need to determine which `/dev/ttyACM*` device maps to which arm.

**Method:** Connect to one port, read a motor position, then physically move one arm and re-read. If the value changes, that port belongs to the arm you moved.

Our result:

| Port | Arm |
|:---|:---|
| `/dev/ttyACM0` | **Follower** |
| `/dev/ttyACM1` | **Leader** |

In [5]:
from lerobot.motors.feetech import FeetechMotorsBus
from lerobot.motors.motors_bus import Motor, MotorNormMode

bus = FeetechMotorsBus(
    port="/dev/ttyACM0",
    motors={
        "motor1": Motor(id=1, model="sts3215", norm_mode=MotorNormMode.DEGREES),
    },
)
bus.connect()

# Read raw position — physically move one arm, then re-run this cell.
# If the value changes, ttyACM0 is that arm.
print("Position:", bus.read("Present_Position", "motor1", normalize=False))

bus.disconnect()

Position: 1986


## 3. Verify Calibration Files

Calibration ensures that the same physical position produces the same numerical value across different arms. Verify that the calibration JSON files are accessible inside the container.

In [6]:
import json
from pathlib import Path

CALIBRATION_DIR = Path("/opt/workspace/lerobot/calibration")

for name in ["my_leader_arm", "my_follower_arm"]:
    fpath = CALIBRATION_DIR / f"{name}.json"
    with open(fpath) as f:
        cal = json.load(f)
    print(f"=== {name} ({fpath}) ===")
    for joint, params in cal.items():
        print(f"  {joint}: id={params['id']}, offset={params['homing_offset']}, "
              f"range=[{params['range_min']}, {params['range_max']}]")
    print()

=== my_leader_arm (/opt/workspace/lerobot/calibration/my_leader_arm.json) ===
  shoulder_pan: id=1, offset=-1990, range=[707, 3437]
  shoulder_lift: id=2, offset=1948, range=[851, 3247]
  elbow_flex: id=3, offset=-1990, range=[840, 3054]
  wrist_flex: id=4, offset=1976, range=[811, 3176]
  wrist_roll: id=5, offset=2030, range=[208, 4025]
  gripper: id=6, offset=1731, range=[1573, 2871]

=== my_follower_arm (/opt/workspace/lerobot/calibration/my_follower_arm.json) ===
  shoulder_pan: id=1, offset=-1595, range=[703, 3425]
  shoulder_lift: id=2, offset=-1760, range=[820, 3221]
  elbow_flex: id=3, offset=1612, range=[862, 3061]
  wrist_flex: id=4, offset=-1398, range=[780, 3180]
  wrist_roll: id=5, offset=2000, range=[119, 3968]
  gripper: id=6, offset=-1466, range=[1430, 2952]



## 4. Run Teleoperation

### Configuration Summary

| Parameter | Follower | Leader |
|:---|:---|:---|
| `type` | `so101_follower` | `so101_leader` |
| `port` | `/dev/ttyACM0` | `/dev/ttyACM1` |
| `id` | `my_follower_arm` | `my_leader_arm` |
| `calibration_dir` | `/opt/workspace/lerobot/calibration` | `/opt/workspace/lerobot/calibration` |

### Calibration Path Override

By default, lerobot 0.5.0 looks for calibration files at:

```
~/.cache/huggingface/lerobot/calibration/robots/{type}/{id}.json          # follower
~/.cache/huggingface/lerobot/calibration/teleoperators/{type}/{id}.json   # leader
```

Since our files are stored under `/opt/workspace/lerobot/calibration/`,
we override the default with `--robot.calibration_dir` and `--teleop.calibration_dir`.

### How to Run

> **Important:** `lerobot-teleoperate` requires interactive terminal input (e.g. pressing Enter to confirm calibration). Jupyter notebook code cells **do not** support this. You must run it in a real terminal.

Open a terminal using one of these methods:

1. **JupyterLab Terminal** — Go to **File → New → Terminal** in the JupyterLab UI
2. **Bash mode** — Change `run.sh` entrypoint from `jupyter` to `bash` and use the container shell

Then paste and run:

```bash
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.calibration_dir=/opt/workspace/lerobot/calibration \
    --teleop.type=so101_leader \
    --teleop.port=/dev/ttyACM1 \
    --teleop.id=my_leader_arm \
    --teleop.calibration_dir=/opt/workspace/lerobot/calibration
```

### Handling the Calibration Prompt

On startup, you may see:

```
Mismatch between calibration values in the motor and the calibration file or no calibration file found
Press ENTER to use provided calibration file associated with the id my_leader_arm,
or type 'c' and press ENTER to run calibration:
```

This means the calibration file **was found**, but the values stored in the motors differ from the file.

> **Press Enter** to write the calibration file values into the motors.
> This is the correct action when using a known good calibration file.
> You may need to press Enter **twice** (once for the leader, once for the follower).

## 5. Verify Success

Once teleoperation is running, you should see log output like:

```
INFO 2026-03-31 12:14:08 so_leader.py:78 my_leader_arm SOLeader connected.
INFO 2026-03-31 12:14:08 follower.py:105 my_follower_arm SOFollower connected.
```

Now try physically moving the **leader arm**. If the **follower arm mirrors the leader's movements in real time**, teleoperation is working correctly.

Press `Ctrl+C` in the terminal to stop teleoperation when you are done.

### Troubleshooting

| Symptom | Possible Cause | Fix |
|:---|:---|:---|
| `Permission denied` on `/dev/ttyACM*` | Missing group permission | Ensure `--group-add dialout` is in `run.sh` |
| `Could not connect on port` | Port not passed to container | Add `--device /dev/ttyACM*` to `run.sh` |
| Follower does not move | Wrong port mapping | Swap `ttyACM0` / `ttyACM1` in the command |
| Jerky or delayed movement | Calibration mismatch | Re-run calibration with `'c'` at the prompt |

### Conclusion

In this lab, you have successfully:

- Identified which USB port maps to the leader and follower arms
- Verified that calibration files are correctly loaded into the motors
- Run real-time teleoperation where the follower mirrors the leader's movements

Teleoperation is the foundation for imitation learning. In the next labs, you will use this setup to record demonstration datasets with `lerobot-record`, train policies (ACT, Diffusion Policy, etc.), and deploy them autonomously on the follower arm.

## Acknowledgements

This lab series builds on the [LeRobot](https://github.com/huggingface/lerobot) project -- the `lerobot-teleoperate`, `lerobot-record`, and `lerobot-train` tools, the **ACT** and **SmolVLA** policies -- and the open-hardware **SO-101** arm from [TheRobotStudio / Hugging Face](https://github.com/TheRobotStudio/SO-ARM100). Training and inference run on AMD Ryzen AI (Radeon gfx1152) with ROCm.


---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
